# Day C7 — 차원 축소, 분류, 통계: spike sorting, 순열 검정, 군집 기반 보정

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/C-neural-signals/day-C7.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/C-neural-signals/day-C7.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 문턱 검출 → PCA → k-means로 두 세포 나누기

In [ ]:
import numpy as np
from scipy.cluster.vq import kmeans2

rng = np.random.default_rng(12)
fs = 30000                                    # 세포외 기록은 보통 수십 kHz
wt = np.arange(-0.5e-3, 1.0e-3, 1 / fs)       # 스파이크 파형 1.5 ms = 45점
def template(depth, width):                   # 아래로 꺼졌다 올라오는 모양
    return -depth * np.exp(-wt ** 2 / (2 * (width * 1e-4) ** 2)) + 0.3 * depth * np.exp(-(wt - 0.4e-3) ** 2 / (2 * (2 * width * 1e-4) ** 2))
A, B = template(80, 1.5), template(55, 3.0)  # 크고 좁은 세포 A, 작고 넓은 세포 B (µV)

# 연속 기록 2초 안에 두 세포의 스파이크를 섞어 넣는다
n = 2 * fs
x = rng.normal(0, 8, n)
truth = []
for tmpl, lab, k in [(A, 0, 60), (B, 1, 60)]:
    for s in rng.choice(np.arange(100, n - 100), k, replace=False):
        x[s:s + len(wt)] += tmpl; truth.append((s + np.argmin(tmpl), lab))
truth = sorted(truth)

# 1) 문턱 검출: 잡음 SD(중앙값 기반)의 -4배 아래로 처음 내려가는 곳
sd = np.median(np.abs(x)) / 0.6745
cross = np.flatnonzero((x[1:] < -4 * sd) & (x[:-1] >= -4 * sd)) + 1
cross = cross[np.insert(np.diff(cross) > 30, 0, True)]        # 1 ms 안의 중복 제거
# 2) 파형 자르기: 최저점을 기준으로 앞 15점, 뒤 30점
peaks = np.array([c + np.argmin(x[c:c + 15]) for c in cross])
W = np.array([x[p - 15:p + 30] for p in peaks])
print("문턱 %.1f µV, 검출 %d개 (실제 %d개)" % (-4 * sd, len(W), len(truth)))

# 3) PCA: 평균을 뺀 뒤 공분산 행렬의 고유분해
Wc = W - W.mean(axis=0)
C = Wc.T @ Wc / (len(W) - 1)
evals, evecs = np.linalg.eigh(C)                  # 오름차순으로 나온다 -> 뒤집기
evals, evecs = evals[::-1], evecs[:, ::-1]
print("PC1–3 설명 분산 비율:", np.round(evals[:3] / evals.sum(), 3))
scores = Wc @ evecs[:, :2]

# 4) k-means로 2개 묶음
_, lab = kmeans2(scores, 2, seed=1, minit="++")
# 실제 정답과 비교 (검출 시점이 ±1 ms 안이면 같은 스파이크로 본다)
t_idx = np.array([t for t, _ in truth]); t_lab = np.array([l for _, l in truth])
match = [t_lab[np.argmin(np.abs(t_idx - p))] if np.min(np.abs(t_idx - p)) < 30 else -1 for p in peaks]
match = np.array(match)
ok = match >= 0
acc = max(np.mean(lab[ok] == match[ok]), np.mean(lab[ok] != match[ok]))   # 묶음 번호는 임의라 두 경우 중 큰 쪽
print("정답과 짝지은 스파이크 %d개, 분류 일치율 %.1f%%" % (ok.sum(), 100 * acc))
for k in [0, 1]:
    print("묶음 %d: %d개, 평균 최저점 %.0f µV" % (k, (lab == k).sum(), W[lab == k].min(axis=1).mean()))

### 예시 2 — 발화율 차이의 순열 검정

In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(13)
# 한 뉴런의 시행별 발화율(Hz): 조건 A 30시행, 조건 B 30시행. 오른쪽으로 꼬리가 긴 분포
a = rng.gamma(shape=2, scale=6, size=30) + 2
b = rng.gamma(shape=2, scale=10, size=30) + 2
obs = b.mean() - a.mean()

data = np.concatenate([a, b])
labels = np.array([0] * len(a) + [1] * len(b))
n_perm = 5000
null = np.empty(n_perm)
for i in range(n_perm):
    lab = rng.permutation(labels)                       # 데이터는 그대로, 조건 이름표만 섞는다
    null[i] = data[lab == 1].mean() - data[lab == 0].mean()

p = (np.sum(np.abs(null) >= abs(obs)) + 1) / (n_perm + 1)   # 양측, +1 보정(0이 나오지 않게)
print("관측 차이 %.2f Hz | 순열 null 평균 %.2f, SD %.2f | 순열 p = %.4f" % (obs, null.mean(), null.std(), p))
print("참고: Welch t-검정 p = %.4f" % stats.ttest_ind(b, a, equal_var=False).pvalue)

# 같은 검정을 다시 돌리면 p가 조금씩 달라진다 (난수 때문). 순열 수가 많을수록 덜 흔들린다
for n in [200, 5000]:
    ps = []
    for _ in range(20):
        nl = np.array([np.diff([data[l == 0].mean(), data[l == 1].mean()])[0] for l in (rng.permutation(labels) for _ in range(n))])
        ps.append((np.sum(np.abs(nl) >= abs(obs)) + 1) / (n + 1))
    print("순열 %4d번 × 20회 반복: p 범위 %.4f – %.4f" % (n, min(ps), max(ps)))

### 예시 3 — 시간-주파수 map: 보정 없음 vs 최대값 보정 vs 군집 보정

In [ ]:
import numpy as np
from scipy import ndimage, stats

rng = np.random.default_rng(14)
n_f, n_t, n_tr = 20, 50, 30                       # 주파수 20칸 × 시간 50칸, 조건마다 30시행
F, T = np.meshgrid(np.arange(n_f), np.arange(n_t), indexing="ij")
blob = 0.6 * np.exp(-((F - 8) ** 2 / (2 * 2.5 ** 2) + (T - 30) ** 2 / (2 * 5 ** 2)))   # 숨긴 효과 덩어리

def maps(n, effect):
    """시행별 시간-주파수 map. 이웃 칸끼리 닮은 잡음(실제 TF map처럼 매끈함)"""
    noise = ndimage.gaussian_filter(rng.normal(0, 1, (n, n_f, n_t)), sigma=(0, 1.5, 3)) * 4
    return noise + (blob if effect else 0)

def tmap(A, B):
    return stats.ttest_ind(B, A, axis=0).statistic

def clusters(t, thr):
    """|t| > thr 인 칸을 이웃끼리 묶고, 묶음마다 t 합(cluster mass)"""
    lab, k = ndimage.label(np.abs(t) > thr)
    return lab, np.array([np.abs(t[lab == i]).sum() for i in range(1, k + 1)])

def analyze(effect, n_perm=500):
    A, B = maps(n_tr, False), maps(n_tr, effect)
    t_obs = tmap(A, B)
    thr = stats.t.ppf(0.975, 2 * n_tr - 2)                 # 칸 하나의 양측 p<0.05 문턱
    lab, mass = clusters(t_obs, thr)
    allm = np.concatenate([A, B]); lbl = np.r_[np.zeros(n_tr), np.ones(n_tr)]
    max_t, max_mass = [], []
    for _ in range(n_perm):
        p = rng.permutation(lbl)
        tp = tmap(allm[p == 0], allm[p == 1])
        max_t.append(np.abs(tp).max())                      # map 하나에서 가장 큰 |t| 하나
        _, m = clusters(tp, thr)
        max_mass.append(m.max() if len(m) else 0)           # map 하나에서 가장 큰 cluster 하나
    pix_thr = np.quantile(max_t, 0.95)
    clu_thr = np.quantile(max_mass, 0.95)
    sig_clu = sum((lab == i + 1).sum() for i, m in enumerate(mass) if m > clu_thr)
    print("효과 %-5s | 보정 없음 %4d칸 | 최대 픽셀 보정 %3d칸 (|t|>%.2f) | cluster 보정 %3d칸" %
          (effect, (np.abs(t_obs) > thr).sum(), (np.abs(t_obs) > pix_thr).sum(), pix_thr, sig_clu))

print("전체 칸 수:", n_f * n_t, "| 숨긴 효과가 최대치의 절반 이상인 칸:", int((blob > 0.3).sum()))
analyze(True)
analyze(False)